# РК-1 по ТМО

**Студент:** Турланов Вячеслав  
**Группа:** ИУ5-66Б  
**Вариант:** 19  

## Задание
Для заданного набора данных произвести:

1. Масштабирование данных для одного числового признака.
2. Преобразование одного категориального признака в количественные двумя способами:
   - `Label Encoding`;
   - `One Hot Encoding`.
3. Построение парных диаграмм для анализа взаимосвязей между признаками.
4. Ответить, какие методы использованы и почему.


## 1. Импорт библиотек и загрузка данных

В работе используется датасет `toy_dataset.csv`, находящийся в архиве `toy_dataset.csv.zip`.


In [1]:
import zipfile
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, OneHotEncoder

pd.set_option('display.max_columns', 20)

zip_path = Path('toy_dataset.csv.zip')
if not zip_path.exists():
    zip_path = Path('/mnt/data/toy_dataset.csv.zip')

with zipfile.ZipFile(zip_path) as z:
    with z.open('toy_dataset.csv') as f:
        data = pd.read_csv(f)

data.head()

,Number,City,Gender,Age,Income,Illness
0,1,Dallas,Male,41,40367.0,No
1,2,Dallas,Male,54,45084.0,No
2,3,Dallas,Male,42,52483.0,No
3,4,Dallas,Male,40,40941.0,No
4,5,Dallas,Male,46,50289.0,No


## 2. Первичный анализ структуры данных

Посмотрим размер таблицы, типы признаков и количество уникальных значений. Это нужно, чтобы выбрать числовой признак для масштабирования и категориальный признак для кодирования.


In [2]:
print('Размер датасета:', data.shape)
print('\nТипы данных:')
print(data.dtypes)
print('\nКоличество уникальных значений:')
print(data.nunique())

Размер датасета: (150000, 6)

Типы данных:
Number       int64
City        object
Gender      object
Age          int64
Income     float64
Illness     object
dtype: object

Количество уникальных значений:
Number     150000
City            8
Gender          2
Age            41
Income      71761
Illness         2
dtype: int64


In [3]:
data.describe(include='all')

,Number,City,Gender,Age,Income,Illness
count,150000.000000,150000,150000,150000.000000,150000.000000,150000
unique,NaN,8,2,NaN,NaN,2
top,NaN,New York City,Male,NaN,NaN,No
freq,NaN,50307,83800,NaN,NaN,137861
mean,75000.500000,NaN,NaN,44.950200,91252.798273,NaN
std,43301.414527,NaN,NaN,11.572486,24989.500948,NaN
min,1.000000,NaN,NaN,25.000000,-654.000000,NaN
25%,37500.750000,NaN,NaN,35.000000,80867.750000,NaN
50%,75000.500000,NaN,NaN,45.000000,93655.000000,NaN
75%,112500.250000,NaN,NaN,55.000000,104519.000000,NaN


## 3. Масштабирование одного числового признака

Для масштабирования выбран признак **`Income`**. Это числовой признак с достаточно большим диапазоном значений, поэтому его удобно привести к единой шкале.

Используем метод **MinMaxScaler**, который преобразует значения признака в диапазон от 0 до 1. Формула масштабирования:

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Такой способ удобен, когда нужно сохранить форму распределения признака, но привести значения к сопоставимому диапазону.


In [4]:
scaler = MinMaxScaler()

data_scaled = data.copy()
data_scaled['Income_MinMax'] = scaler.fit_transform(data_scaled[['Income']])

data_scaled[['Income', 'Income_MinMax']].head(10)

,Income,Income_MinMax
0,40367.0,0.230700
1,45084.0,0.257228
2,52483.0,0.298840
3,40941.0,0.233928
4,50289.0,0.286501
5,50786.0,0.289296
6,33155.0,0.190140
7,30914.0,0.177537
8,68667.0,0.389858
9,50082.0,0.285337


In [5]:
data_scaled[['Income', 'Income_MinMax']].agg(['min', 'max', 'mean', 'std'])

,Income,Income_MinMax
min,-654.000000,0.000000
max,177157.000000,1.000000
mean,91252.798273,0.516879
std,24989.500948,0.140540


### Вывод по масштабированию

После применения `MinMaxScaler` исходный признак `Income` был преобразован в новый признак `Income_MinMax`. Минимальное значение стало равно 0, максимальное — 1. Это полезно для алгоритмов машинного обучения, чувствительных к масштабу признаков, например для методов на основе расстояний, градиентных методов и нейронных сетей.


## 4. Преобразование категориального признака методом Label Encoding

Для кодирования выбран категориальный признак **`City`**. В нем 8 уникальных городов.

Метод **Label Encoding** заменяет каждую категорию целым числом. Например, одному городу соответствует 0, другому — 1 и так далее.


In [6]:
label_encoder = LabelEncoder()

data_label_encoded = data_scaled.copy()
data_label_encoded['City_LabelEncoded'] = label_encoder.fit_transform(data_label_encoded['City'])

city_mapping = pd.DataFrame({
    'City': label_encoder.classes_,
    'City_LabelEncoded': range(len(label_encoder.classes_))
})

city_mapping

,City,City_LabelEncoded
0,Austin,0
1,Boston,1
2,Dallas,2
3,Los Angeles,3
4,Mountain View,4
5,New York City,5
6,San Diego,6
7,Washington D.C.,7


In [7]:
data_label_encoded[['City', 'City_LabelEncoded']].head(10)

,City,City_LabelEncoded
0,Dallas,2
1,Dallas,2
2,Dallas,2
3,Dallas,2
4,Dallas,2
5,Dallas,2
6,Dallas,2
7,Dallas,2
8,Dallas,2
9,Dallas,2


### Вывод по Label Encoding

`Label Encoding` прост и компактен: один категориальный столбец превращается в один числовой столбец. Однако для признака `City` этот метод может быть неидеальным, потому что модель может ошибочно интерпретировать номера городов как порядок или расстояние между категориями. Например, код 7 не означает, что соответствующий город «больше» города с кодом 1.


## 5. Преобразование категориального признака методом One Hot Encoding

Теперь преобразуем тот же признак **`City`** методом **One Hot Encoding**.

Метод создает отдельный бинарный столбец для каждой категории. Если объект относится к конкретному городу, в соответствующем столбце стоит 1, в остальных — 0.


In [8]:
try:
    encoder = OneHotEncoder(sparse_output=False)
except TypeError:
    # Для старых версий scikit-learn
    encoder = OneHotEncoder(sparse=False)

city_ohe = encoder.fit_transform(data[['City']])
city_ohe_columns = encoder.get_feature_names_out(['City'])

city_ohe_df = pd.DataFrame(city_ohe, columns=city_ohe_columns, index=data.index).astype(int)

data_one_hot_encoded = pd.concat([data_label_encoded, city_ohe_df], axis=1)

city_ohe_df.head(10)

,City_Austin,City_Boston,City_Dallas,City_Los Angeles,City_Mountain View,City_New York City,City_San Diego,City_Washington D.C.
0,0,0,1,0,0,0,0,0
1,0,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,0,0,1,0,0,0,0,0
4,0,0,1,0,0,0,0,0
5,0,0,1,0,0,0,0,0
6,0,0,1,0,0,0,0,0
7,0,0,1,0,0,0,0,0
8,0,0,1,0,0,0,0,0
9,0,0,1,0,0,0,0,0


In [9]:
data_one_hot_encoded[['City'] + list(city_ohe_columns)].head(10)

,City,City_Austin,City_Boston,City_Dallas,City_Los Angeles,City_Mountain View,City_New York City,City_San Diego,City_Washington D.C.
0,Dallas,0,0,1,0,0,0,0,0
1,Dallas,0,0,1,0,0,0,0,0
2,Dallas,0,0,1,0,0,0,0,0
3,Dallas,0,0,1,0,0,0,0,0
4,Dallas,0,0,1,0,0,0,0,0
5,Dallas,0,0,1,0,0,0,0,0
6,Dallas,0,0,1,0,0,0,0,0
7,Dallas,0,0,1,0,0,0,0,0
8,Dallas,0,0,1,0,0,0,0,0
9,Dallas,0,0,1,0,0,0,0,0


### Вывод по One Hot Encoding

`One Hot Encoding` лучше подходит для номинальных категориальных признаков, где нет естественного порядка между категориями. Признак `City` именно такой: города нельзя корректно упорядочить по смыслу. Поэтому для большинства моделей машинного обучения `One Hot Encoding` будет более корректным вариантом, чем `Label Encoding`. Недостаток метода — увеличение числа столбцов, особенно если категорий очень много.


## 6. Итоговый фрагмент преобразованного датасета

Ниже показаны исходные признаки и новые признаки, полученные после масштабирования и кодирования.


In [10]:
result_columns = ['Income', 'Income_MinMax', 'City', 'City_LabelEncoded'] + list(city_ohe_columns)
data_one_hot_encoded[result_columns].head(10)

,Income,Income_MinMax,City,City_LabelEncoded,City_Austin,City_Boston,City_Dallas,City_Los Angeles,City_Mountain View,City_New York City,City_San Diego,City_Washington D.C.
0,40367.0,0.230700,Dallas,2,0,0,1,0,0,0,0,0
1,45084.0,0.257228,Dallas,2,0,0,1,0,0,0,0,0
2,52483.0,0.298840,Dallas,2,0,0,1,0,0,0,0,0
3,40941.0,0.233928,Dallas,2,0,0,1,0,0,0,0,0
4,50289.0,0.286501,Dallas,2,0,0,1,0,0,0,0,0
5,50786.0,0.289296,Dallas,2,0,0,1,0,0,0,0,0
6,33155.0,0.190140,Dallas,2,0,0,1,0,0,0,0,0
7,30914.0,0.177537,Dallas,2,0,0,1,0,0,0,0,0
8,68667.0,0.389858,Dallas,2,0,0,1,0,0,0,0,0
9,50082.0,0.285337,Dallas,2,0,0,1,0,0,0,0,0


## 7. Парные диаграммы

Для дополнительного анализа построим **парные диаграммы** (`pairplot`). Они позволяют увидеть распределения отдельных числовых признаков на диагонали и попарные зависимости между признаками вне диагонали.

Так как исходный датасет содержит 150 000 строк, для визуализации используется случайная выборка. Это ускоряет построение графиков и делает диаграммы читаемыми, при этом сохраняет общее представление о структуре данных.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Выберем числовые признаки, которые имеют содержательный смысл для попарного анализа.
# Признак Number является идентификатором строки, поэтому его не используем.
pairplot_features = ['Age', 'Income', 'Income_MinMax']

pairplot_sample = data_one_hot_encoded[pairplot_features + ['City']].sample(
    n=1000,
    random_state=19
)

sns.pairplot(pairplot_sample[pairplot_features], diag_kind='hist')
plt.suptitle('Парные диаграммы для числовых признаков', y=1.02)
plt.show()


Для более наглядной интерпретации построим второй вариант парных диаграмм с раскраской объектов по категориальному признаку `City`. Это помогает оценить, различаются ли распределения числовых признаков для разных городов.


In [ ]:
sns.pairplot(
    pairplot_sample,
    vars=pairplot_features,
    hue='City',
    diag_kind='hist',
    plot_kws={'alpha': 0.6}
)
plt.suptitle('Парные диаграммы с раскраской по признаку City', y=1.02)
plt.show()


### Вывод по парным диаграммам

Парные диаграммы использованы как метод визуального анализа данных. Они удобны, потому что позволяют одновременно оценить:

1. распределение каждого числового признака;
2. наличие или отсутствие заметной зависимости между парами признаков;
3. различия между группами объектов при раскраске по категориальному признаку.

В данной работе парные диаграммы построены для признаков `Age`, `Income` и `Income_MinMax`. Признак `Income_MinMax` является масштабированной версией `Income`, поэтому их взаимосвязь на графике должна быть линейной. Это подтверждает корректность масштабирования методом `MinMaxScaler`.


## Ответ на вопрос: какие методы использованы и почему?

В работе были использованы следующие методы предварительной обработки данных:

1. **Масштабирование MinMaxScaler для признака `Income`.**  
   Этот метод был выбран, потому что `Income` является числовым признаком с большим диапазоном значений. `MinMaxScaler` приводит значения к диапазону от 0 до 1, что делает признак более удобным для алгоритмов, чувствительных к масштабу данных.

2. **Label Encoding для признака `City`.**  
   Метод заменяет текстовые категории числовыми метками. Он прост, не увеличивает размерность данных и может быть полезен для алгоритмов, которые умеют работать с условно-категориальными кодами. Но для номинального признака `City` есть риск появления ложного порядка между городами.

3. **One Hot Encoding для признака `City`.**  
   Метод преобразует каждую категорию в отдельный бинарный признак. Он выбран как более корректный способ кодирования признака `City`, потому что города являются номинальными категориями без естественного порядка. Такой подход исключает ложную зависимость между числовым кодом и смыслом категории.

**Общий вывод:** для масштабирования числового признака `Income` целесообразно использовать `MinMaxScaler`, а для номинального категориального признака `City` более корректным является `One Hot Encoding`. `Label Encoding` также был выполнен для сравнения, но для городов он менее предпочтителен, так как задает искусственный порядок категорий.
